### Basic Import

In [2]:
# Importing necessary libraries
import numpy as np        # For numerical operations
import pandas as pd       # For data manipulation and analysis
import matplotlib.pyplot as plt  # For data visualization
%matplotlib inline

# Importing WordCloud for text visualization
from wordcloud import WordCloud

# Importing NLTK for natural language processing
import nltk
from nltk.corpus import stopwords    # For stopwords


# Downloading NLTK data
nltk.download('stopwords')   # Downloading stopwords data
nltk.download('punkt')       # Downloading tokenizer data

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\patil\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\patil\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [3]:
# Read CSV file
df = pd.read_csv("spam.csv")

# Display first 5 rows of dataset
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [5]:
#Dropig NaN colums
df.drop(columns=['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], inplace=True)
df.head()

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [6]:
# renaming colums for ease of understanding
df.rename(columns = { 'v1' : 'target', 'v2' : 'text' }, inplace = True)
df.head()  

,target,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


### Data Preprocessing

In [7]:
from sklearn.preprocessing import LabelEncoder 

encoder = LabelEncoder()

df['target'] = encoder.fit_transform(df['target'])

df.head

<bound method NDFrame.head of       target                                               text
0          0  Go until jurong point, crazy.. Available only ...
1          0                      Ok lar... Joking wif u oni...
2          1  Free entry in 2 a wkly comp to win FA Cup fina...
3          0  U dun say so early hor... U c already then say...
4          0  Nah I don't think he goes to usf, he lives aro...
...      ...                                                ...
5567       1  This is the 2nd time we have tried 2 contact u...
5568       0              Will Ì_ b going to esplanade fr home?
5569       0  Pity, * was in mood for that. So...any other s...
5570       0  The guy did some bitching but I acted like i'd...
5571       0                         Rofl. Its true to its name

[5572 rows x 2 columns]>

In [8]:
#check duplicates
df.duplicated().sum()

np.int64(403)

In [9]:
len(df)

5572

In [10]:
#remove duplicates
df = df.drop_duplicates(keep= 'first')
len(df)

5169

### Feature Engineering

In [11]:
#importing Porter Stemmer for text stemming
from nltk.stem.porter import PorterStemmer

#importing string module for handling special character
import string 

#creating instance for PorterStemmer
ps = PorterStemmer()


In [13]:
#lower case transformation and text processing function
def transform_text(text):
    #transform text to lower case
    text = text.lower()

    #tokenize using nltk
    text = nltk.word_tokenize(text)

    #removing special character
    y = []
    for i in text:
        if i.isalnum():
            y.append(i)

    
    #removing stopwordas and punctuation
    text = y[:]
    y.clear()

    #loop through the tokens to remove stopwords and punctuations
    for i in text:
        if i not in stopwords.words('english') and i not in string.punctuation:
            y.append(i)

    #Stemming using PorterStemmer
    text = y[:]
    y.clear()
    for i in text:
        y.append(ps.stem(i))

    #join the processed token back to single string 
    return " ".join(y)

In [14]:
transform_text('Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...')

'go jurong point crazi avail bugi n great world la e buffet cine got amor wat'

In [15]:
df['transformed_text'] = df['text'].apply(transform_text)
df.head()

C:\Users\patil\AppData\Local\Temp\ipykernel_21644\568599122.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['transformed_text'] = df['text'].apply(transform_text)


,target,text,transformed_text
0,0,"Go until jurong point, crazy.. Available only ...",go jurong point crazi avail bugi n great world...
1,0,Ok lar... Joking wif u oni...,ok lar joke wif u oni
2,1,Free entry in 2 a wkly comp to win FA Cup fina...,free entri 2 wkli comp win fa cup final tkt 21...
3,0,U dun say so early hor... U c already then say...,u dun say earli hor u c alreadi say
4,0,"Nah I don't think he goes to usf, he lives aro...",nah think goe usf live around though


In [16]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
tfid = TfidfVectorizer(max_features=500)

In [17]:
X = tfid.fit_transform(df['transformed_text']).toarray()
y = df['target'].values

### Train Test Split

In [18]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.20, random_state=2)

### Model Training 

In [19]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier

In [49]:
svc = SVC(kernel="sigmoid", gamma= 1.0)
knc = KNeighborsClassifier()
mnb = MultinomialNB()
dtc = DecisionTreeClassifier(max_depth= 5)
lrc = LogisticRegression(solver='liblinear', penalty='l1')
rfc = RandomForestClassifier(n_estimators=50, random_state= 2)
abc = AdaBoostClassifier(n_estimators=50, random_state=2)
bc = BaggingClassifier(n_estimators=50, random_state=2)
etc = ExtraTreesClassifier(n_estimators=50, random_state=2)
gbdt = GradientBoostingClassifier(n_estimators=50, random_state=2)
xgb = XGBClassifier(n_estimators = 50 , random_state = 2)

In [50]:
clfs = {
    'SVC': svc,
    'KNC': knc,
    'MNB': mnb,
    'DTC': dtc,
    'LRC': lrc,
    'RFC': rfc,
    'Adaboost': abc,
    'BGC': bc,
    'ETC': etc,
    'GBDT': gbdt,
    'XGB': xgb
}

### Model Evaluation

In [51]:
from sklearn.metrics import accuracy_score, precision_score
def train_classifier(clf, X_train, y_train, X_test, y_test):
    clf.fit(X_train,y_train)
    y_pred = clf.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    return accuracy, precision

In [52]:
accuracy_score_list = []
precision_score_list = []
for name , clf in clfs.items():
    current_accuracy, current_precision = train_classifier(clf, X_train, y_train, X_test, y_test)
    print()
    print("For: ", name)
    print("Accuracy: ", current_accuracy)
    print("Precision: ", current_precision)

    accuracy_score_list.append(current_accuracy)
    precision_score_list.append(current_precision)


For:  SVC
Accuracy:  0.9661508704061895
Precision:  0.9327731092436975

For:  KNC
Accuracy:  0.9274661508704062
Precision:  1.0

For:  MNB
Accuracy:  0.9709864603481625
Precision:  0.9655172413793104

For:  DTC
Accuracy:  0.9381044487427466
Precision:  0.9021739130434783

For:  LRC
Accuracy:  0.9622823984526112
Precision:  0.9541284403669725

For:  RFC
Accuracy:  0.9738878143133463
Precision:  0.9586776859504132

For:  Adaboost
Accuracy:  0.9245647969052224
Precision:  0.875

For:  BGC
Accuracy:  0.965183752417795
Precision:  0.9180327868852459

For:  ETC
Accuracy:  0.9758220502901354
Precision:  0.9448818897637795

For:  GBDT
Accuracy:  0.9526112185686654
Precision:  0.9405940594059405

For:  XGB
Accuracy:  0.9709864603481625
Precision:  0.95
